# Managing your Seamless workflow Spider

Seamless is a nice name, but it does require some work!
In this notebook I will outline the necessities and provide you, the user, the Python versions of cli commands that are just needed to run your workflow on a super computer.
This workflow consists of jupyter notebooks and is nicely structured and when the workflow is done the code will return the executed notebooks.
We use a Python library called Papermill.

This explanation will consist of 3 parts:
1. File structure
2. Access to any SLURM based HPC
3. Required scripts
4. Getting your notebooks ready for HPC use with Papermill
5. How to communicate with the HPC

Also some quircks:
- use `display(f"print")` instead of `print(f"print")`

## 1. File Structure

This is the file structure used to make seamless work nicely:

- seamless (your research)
    - notebooks
        - contains your workflow in structured notebooks
    - regions
        - folders with different countries
            - folders with different regions based upon country
                - Contain the **logs** (errors and output files) and executed notebooks when done
    - scripts
        - three scripts that will run the workflow remotely
    - done (optional, but strongly recommended)
        - folders with different counties
            - contains file to check what runs are done already

### notebooks

The notebooks you want to run are stored in this folder.

### regions

This folder contains the subfolders countries, which contain subfolders of the regions you want to run.
The names of the subfolders will be used in the example as the region_id for the Notebooks.
The name of the subfolder, for example 'region1' will be passed through to the Python script to the notebooks.
So:
- regions
    - country1
        - region_1
            - output
            - error_file.err
            - output_file.out
            - settings.json
        - region_2
        - etc.
    - country2
        - region_x

### scripts

See 'required scripts'.

### done

Recommended to have! A folder with a .csv and a csv.lock file to check whether some runs are already done before. Removing unnecessary computation time.

## 2. Access to any SLURM based HPC

Ask eWaterCycle admin to help you with tailoring your seamless workflow. We can currently use Snellius and Spider. DelftBlue would also be easy to work with.

## 3. Required scripts

I will use our first succesful trial namescheme, we did climate change impact analysis, so in short: cci.
As mentioned there are 3 scripts:

- A bash script to submit jobs on HPC
    - `submit_cci.sh`
- A bash script that configures SLURM
    - `run_cci.slurm`
- A Python script that will run your workflow
    - `cci.py`
 


### Submit script (old)

#### A simple bash script

This a simple bash (.sh) script that will loop through your folders of interest in the region folder.
It looks like this:

```
#!/bin/bash

for region in $(ls regions)
do
    sbatch --job-name=$region --error=logs/$region.err --output=logs/$region.out --export=REGION_ID=$region scripts/run_cci.slurm
    sleep 1
done
```

So let me walk through it!
1. `#!/bin/bash` this lets the system know to use bash to execute this file with.
2. `for region in $(ls regions)` is our for loop. It runs the `ls` command on the 'regions' folder, so it will list all the subfolders in there that we made. I take the name of the subfolder with `$` and call it `region`
3. `do` is the start of the for loop.
4. `sbatch` is the command to queue your jobs on a SLURM based supercomputer. Then there are some arguments:
    - `--job-name=$region` sets the job name to the region
    - `--error=logs/$region.err --output=logs/$region.out` set the output file and error file which are written with every SLURM job to a place in the logs with the same name as the region. These are **VERY IMPORTANT** as they will quickly let you know what is wrong with the run.
    - `--export=REGION_ID=$region` is also important, it provides an environmental variable with the job called: 'REGION_ID' which is used to run the correct region with the Python script
    - `scripts/run_cci.slurm` is the actual slurm job that will run, see next section for that.
5. `sleep 1` means that it will take a interval of 1 second to submit the slurm job, this is just a safety precaution for simultaneously accessing files which sometimes go wrong on HPC systems.
6. `done` completes the loop iteration.

#### Bash Script Spider

```
#!/bin/bash

for country in regions/*
do
    country_name=$(basename "$country")

    for region in "$country"/*
    do
        region_name=$(basename "$region")

        sbatch \
            --job-name="$region_name" \
            --error="regions/$country_name/$region_name/$region_name.err" \
            --output="regions/$country_name/$region_name/$region_name.out" \
            --export=REGION_ID="$region_name",COUNTRY="$country_name" \
            scripts/run_cci.slurm

        break 2  # done as testing to see if this works
        # sleep 1  # for small workloads use this
    done
done
```

Which is basically the same as above but now loops over the countries as well and passes the country to the slurm script. Note that the error and output files are changed as well. The break is there to only do the first region of the first country for testing purposes.

### SLURM script (Snellius)

The SLURM script (.slurm) is used to run on a SLURM based supercomputer (who would have guessed).
The one we used for cci is the following:

```
#!/bin/bash
#SBATCH -t 04:00:00
#SBATCH -p rome
#SBATCH --cpus-per-task=1
#SBATCH --mem=16G

module load 2025
module spider Mamba/24.9.0-0
conda init
source ~/.bashrc

conda activate ewatercycle_snellius

mkdir /scratch-shared/mmelotto/climate_data
rclone mount --read-only --allow-non-empty dcache_shared:/climate-data/obs6 /scratch-shared/mmelotto/climate_data &

mkdir /home/mmelotto/climate_data
rclone mount --read-only --allow-non-empty dcache_shared:/climate-data/obs6 /home/mmelotto/climate_data &

mkdir /scratch-shared/mmelotto/caravan_data
rclone mount --read-only --allow-non-empty dcache_shared:/climate-data/caravan /scratch-shared/mmelotto/caravan_data &

cd /gpfs/home2/mmelotto/seamless/climatechangeimpact

python scripts/cci.py $REGION_ID
```

Again I will walk through it:
1. `#!/bin/bash` tells the system what executable to use for this script
2. `#SBATCH` defines certain parameters for the 'batch job' (this is what you call running your scripts on a HPC system), we already discussed the job name, error & output files
   - `-t 04:00:00` means that the runtime of the job is 4 hours, 0 minutes and 0 seconds.
   - `-p rome` is the [computenode of Snellius](https://servicedesk.surf.nl/wiki/spaces/WIKI/pages/30660208/Snellius+hardware) that this batch job will run on.
   - `--cpus-per-task=1` defines how many cpus you need per task.
   - `--mem=16G` how much memory needs to be allocated per task.
3. `module load 2025` and `module spider Mamba/24.9.0-0` loads the modules stacks that there are on Snellius. From which it loads spider and Mamba
4. `conda init` we need to initialize conda for the eWaterCycle environment to work later
5. `source ~/.bashrc` reloads your Bash configuration into the current shell so that changes like new environment variables, aliases, or functions take effect immediately without opening a new terminal.
6. `conda activate ewatercycle_snellius` activates the environment I created on my account for the eWaterCycle software packages to run in.
7. The `mkdir` and `rclone` commands are there to make sure we connect our dCache storage, which contains a lot of hydrological data, to our jobs.
8. `cd /gpfs/home2/mmelotto/seamless/climatechangeimpact` moves us into our cci directory
9. `python scripts/cci.py $REGION_ID` runs the Python script that will host the running of the notebooks. It gives the argument REGION_ID so that the Python script knows what region to set.

### SLURM script (Spider)

```
#!/bin/bash
#SBATCH -N 1
#SBATCH -c 2
#SBATCH -t 04:00:00
#SBATCH -p short

conda init
source ~/.bashrc
conda activate ewatercycle_spider

mkdir /project/ewater/Data/climate_data
rclone mount --read-only --allow-non-empty dcache:/climate-data/obs6 /project/ewater/Data/climate_data &

mkdir /home/ewater-mmelotto/climate_data
rclone mount --read-only --allow-non-empty dcache:/climate-data/obs6 /home/ewater-mmelotto/climate_data &

mkdir /project/ewater/Data/caravan_data
rclone mount --read-only --allow-non-empty dcache:/climate-data/caravan /project/ewater/Data/caravan_data &

cd /home/ewater-mmelotto/seamless_spider/climatechangeimpact

python scripts/cci.py $REGION_ID $COUNTRY
```

### Python script

## 4. Getting your notebooks ready for HPC use with Papermill

We are using this seamless integration to run a lot of notebooks all with slight changes from one another.
These slight changes are done via Papermill.

This is a vital part for getting your Notebooks to run different regions/calibrations per run.
So in this example we change the region_id per run.
So every job runs a different regions to do this, we will need to make a `parameters` code cell with the **tag** 'parameters'.
When the main Python script calls the notebooks into action it will so with an argument for this region_id. (or your specific usecase of course)

This will look like this:

In [ ]:
# Parameters for Papermill
region_id = None

Note that I say it will look like that!
You will have to something more: you need to [give the cell a tag](https://papermill.readthedocs.io/en/latest/usage-parameterize.html): `parameters`

Assuming you are doing this from an eWaterCycle machine: go to the topright in Jupyter Hub, click on the cogs --> common tools --> select your code cell --> give it the tag 'parameters'
If you are doing it somewhere else, please refer to the [official documentation](https://papermill.readthedocs.io/en/latest/usage-parameterize.html).

It will then look like the next cell, so visually nothing changes, but the cell now has a **tag**.

In [ ]:
# Parameters for Papermill
region_id = None

The example we used for the climate change impact analysis is this:

In [ ]:
# Parameters
region_id = None
settings_path = "settings.json"

So when run on HPC it becomes this for example:

In [ ]:
# Parameters
country = "australia"
region_id = "camelsaus_102101A"
settings_path = "regions/australia/camelsaus_102101A/settings.json"

## 5. How to communicate with the HPC

We will be using ssh and rsync, which normally will be done in command line, but can also be done via Python

### rsync

We have to get our folders and files to the HPC.
We will do this using the `rsync` command.
This syncs our source folder with the destination folder on the HPC.

Here follows an example of the rsync we ran.

In [1]:
from pathlib import Path
import pexpect
import os
from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv()
load_dotenv(env_path)

PASSWORD = os.getenv("PASSWORD_SPIDER")
username = os.getenv("USERNAME_SPIDER")

home = Path.home()
source = str(home / "seamless_spider/")
destination = f"{username}@spider.surfsara.nl:/home/{username}/"

print(f"{source = }")
print(f"{destination = }")

source = '/home/mmelotto/seamless_spider'
destination = 'ewater-mmelotto@spider.surfsara.nl:/home/ewater-mmelotto/'


In [ ]:
child = pexpect.spawn(f"rsync -avh {source} {destination}")
# child.expect("Password:")
# child.sendline(PASSWORD)
child.wait()

# Get everything the child printed
child.expect(pexpect.EOF)  # read until the process ends
output = child.before
print(output.decode("utf-8"))

### rsync your results back

Now when we are done we also want to sync the results back!
Or mostly the error logs...

In [2]:
# Results
child = pexpect.spawn(f"rsync -avh {destination}/seamless_spider/climatechangeimpact/ {str(home)}/seamless_spider/climatechangeimpact")
# child.expect("Password:")
# child.sendline(PASSWORD)
child.wait()

# Get everything the child printed
child.expect(pexpect.EOF)  # read until the process ends
output = child.before
print(output.decode("utf-8"))

# Logs
# child = pexpect.spawn(f"rsync -avh {destination}/seamless_spider/climatechangeimpact/logs {str(home)}/results_seamless")
# child.expect("Password:")
# child.sendline(PASSWORD)
# child.wait()

# Get everything the child printed
child.expect(pexpect.EOF)  # read until the process ends
output = child.before
print(output.decode("utf-8"))

receiving incremental file list
./
__pycache__/
__pycache__/cara.cpython-312.pyc
done/australia/regions_done.csv
done/australia/regions_done.csv.lock
regions/australia/camelsaus_102101A/
regions/australia/camelsaus_102101A/available_climate_datasets.json
regions/australia/camelsaus_102101A/camelsaus_102101A.err
regions/australia/camelsaus_102101A/camelsaus_102101A.json
regions/australia/camelsaus_102101A/camelsaus_102101A.out
regions/australia/camelsaus_102101A/settings.json
regions/australia/camelsaus_102101A/step_0a_select_caravan_region_time_and_scenarios_executed.ipynb
regions/australia/camelsaus_102101A/step_0b_select_CMIP_forcing_executed.ipynb
regions/australia/camelsaus_102101A/step_1a_generate_historical_forcing_executed.ipynb
regions/australia/camelsaus_102101A/step_1b_generate_future_forcing_executed.ipynb
regions/australia/camelsaus_102101A/step_1c_generate_DestinE_future_forcing_executed.ipynb
regions/australia/camelsaus_102101A/step_2a_calibrate_HBV_SCE_executed.ipynb
reg

### Submitting your job

For this we will have to create a ssh (secure shell) connection

In [12]:
import paramiko
import time
import sys

In [15]:
hostname = "spider.surfsara.nl"
# username = "mmelotto"

destination_script = f"/home/{username}/seamless_spider/climatechangeimpact"

ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(
    hostname,
    username=username,
    password=PASSWORD,
    look_for_keys=False,
    allow_agent=False
)


cmd = """
cd seamless_spider/climatechangeimpact && \
ls && \
bash scripts/submit_cci.sh
timeout 30 watch -n 1 squeue
"""

stdin, stdout, stderr = ssh.exec_command(cmd)
output = stdout.read().decode()
error = stderr.read().decode()

# Get outputs
out = stdout.read().decode('utf-8')
err = stderr.read().decode('utf-8')

ssh.close()

print(f"{out = }")
print(f"{err = }")

BadAuthenticationType: Bad authentication type; allowed types: ['publickey']

Now let us look at our output to check if the scripts are coming through!

In [ ]:
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(
    hostname,
    username=username,
    password=PASSWORD,
    look_for_keys=False,
    allow_agent=False
)

# open a channel manually for streaming
transport = ssh.get_transport()
channel = transport.open_session()

# run the watch command
channel.exec_command("watch -n 1 squeue")

start = time.time()
duration = 1 * 60    # 1 minute in seconds

print("Streaming remote output...\n")

try:
    while True:
        if channel.recv_ready():
            data = channel.recv(4096).decode("utf-8", errors="ignore")
            print(data, end="")  # print live in notebook
            
        if time.time() - start > duration:
            print("\n\n--- Time limit reached, stopping watch ---")
            channel.close()
            break

        if channel.exit_status_ready():  # watch stopped?
            break

        time.sleep(0.2)

finally:
    print("Stopping remote output...")
    ssh.close()